In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool, CustomJS, TextInput
from bokeh.layouts import column
from bokeh.transform import linear_cmap
import bokeh
from bokeh.io import show


import polars as pl
import numpy as np
from scipy.sparse import coo_matrix
import umap

In [2]:
bokeh.io.output_notebook()

Loading BokehJS ...

In [3]:
def interactive_from_embeddings(
    embedding,
    labels=None,
    values=None,
    hover_data=None,
    cmap="Blues",
    color_key_cmap="Spectral",
    background="white",
    width=800,
    height=800,
    point_size=None,
    tools=None,
    alpha=None,
):
    """
    Create an interactive Bokeh plot directly from 2D embeddings
    WITHOUT requiring a UMAP model object.
    """

    embedding = np.asarray(embedding)
    if embedding.shape[1] != 2:
        raise ValueError("Embedding must be of shape (n_samples, 2)")

    n = embedding.shape[0]
    if point_size is None:
        point_size = 100.0 / np.sqrt(n)

    df = pd.DataFrame(
        {
            "x": embedding[:, 0],
            "y": embedding[:, 1],
        }
    )

    # ----- Coloring logic (labels OR values OR constant) -----

    if labels is not None and values is not None:
        raise ValueError("Choose only one of labels or values")

    if labels is not None:
        df["label"] = labels
        unique_labels = np.unique(labels)
        cmap_obj = plt.get_cmap(color_key_cmap)
        palette = [
            bokeh.colors.RGB(*[int(c * 255) for c in cmap_obj(i)[:3]]).to_hex()
            for i in np.linspace(1, 0, len(unique_labels))  # blue low to red high
        ]
        color_map = dict(zip(unique_labels, palette))
        df["color"] = pd.Series(labels).map(color_map)

        color_spec = "color"

    elif values is not None:
        df["value"] = values
        cmap_obj = plt.get_cmap(cmap)
        palette = [
            bokeh.colors.RGB(*[int(c * 255) for c in cmap_obj(i)[:3]]).to_hex()
            for i in np.linspace(0, 1, 256)
        ]

        color_spec = linear_cmap(
            field_name="value",
            palette=palette,
            low=np.min(values),
            high=np.max(values),
        )

    else:
        # single mid-color
        color_spec = plt.get_cmap(cmap)(0.5)
        color_spec = bokeh.colors.RGB(
            int(color_spec[0] * 255),
            int(color_spec[1] * 255),
            int(color_spec[2] * 255),
        ).to_hex()

    # hover_data columns
    if hover_data is not None:
        for col in hover_data.columns:
            df[col] = hover_data[col]

    # alpha channel
    if alpha is None:
        df["alpha"] = 1.0
    else:
        df["alpha"] = alpha

    source = ColumnDataSource(df)

    # ----- Create Bokeh figure -----

    fig = figure(
        width=width,
        height=height,
        background_fill_color=background,
        tools=tools if tools is not None else "pan,wheel_zoom,box_zoom,save,reset,help",
        tooltips=[
            (col, f"@{{{col}}}")
            for col in (hover_data.columns if hover_data is not None else [])
        ],
    )

    fig.scatter(
        x="x",
        y="y",
        source=source,
        color=color_spec,
        size=point_size,
        alpha="alpha",
    )

    fig.grid.visible = False
    fig.axis.visible = False

    return fig

In [4]:
umap_df = pl.scan_parquet(
    "/zata/zippy/kresgeb/nmf_stuff/dlPFC/output/nmf_umaps/umap_embeddings/first_save.parquet"
).collect()
print(umap_df)

shape: (41_250, 9)
┌────────┬─────────┬────────┬───────────┬───┬─────┬─────────┬───────────┬───────────┐
│ run_id ┆ pattern ┆ seed   ┆ tol       ┆ … ┆ k   ┆ row_idx ┆ UMAP1     ┆ UMAP2     │
│ ---    ┆ ---     ┆ ---    ┆ ---       ┆   ┆ --- ┆ ---     ┆ ---       ┆ ---       │
│ i32    ┆ i32     ┆ i32    ┆ f64       ┆   ┆ i64 ┆ u32     ┆ f32       ┆ f32       │
╞════════╪═════════╪════════╪═══════════╪═══╪═════╪═════════╪═══════════╪═══════════╡
│ 649    ┆ 64      ┆ 2025   ┆ 0.00001   ┆ … ┆ 90  ┆ 0       ┆ 10.672416 ┆ 7.081692  │
│ 452    ┆ 9       ┆ 42     ┆ 0.00001   ┆ … ┆ 20  ┆ 1       ┆ 5.647392  ┆ -1.078495 │
│ 640    ┆ 42      ┆ 120301 ┆ 0.00001   ┆ … ┆ 100 ┆ 2       ┆ 7.034378  ┆ 11.608908 │
│ 718    ┆ 2       ┆ 1029   ┆ 0.0000001 ┆ … ┆ 80  ┆ 3       ┆ -0.695964 ┆ 10.093839 │
│ 715    ┆ 50      ┆ 1029   ┆ 0.0000001 ┆ … ┆ 50  ┆ 4       ┆ 14.760114 ┆ 15.162823 │
│ …      ┆ …       ┆ …      ┆ …         ┆ … ┆ …   ┆ …       ┆ …         ┆ …         │
│ 139    ┆ 25      ┆ 120301 ┆ 0.000

In [5]:
umap_df2 = umap_df.filter(pl.col("k")==100)

plot = interactive_from_embeddings(
    embedding=umap_df2[["UMAP1", "UMAP2"]].to_numpy(),
    labels=umap_df2["pattern"],
    hover_data=umap_df2.drop("UMAP1", "UMAP2"),
    point_size=3,
)

show(plot)

In [6]:
print(umap_df2)

shape: (7_500, 9)
┌────────┬─────────┬────────┬───────────┬───┬─────┬─────────┬───────────┬───────────┐
│ run_id ┆ pattern ┆ seed   ┆ tol       ┆ … ┆ k   ┆ row_idx ┆ UMAP1     ┆ UMAP2     │
│ ---    ┆ ---     ┆ ---    ┆ ---       ┆   ┆ --- ┆ ---     ┆ ---       ┆ ---       │
│ i32    ┆ i32     ┆ i32    ┆ f64       ┆   ┆ i64 ┆ u32     ┆ f32       ┆ f32       │
╞════════╪═════════╪════════╪═══════════╪═══╪═════╪═════════╪═══════════╪═══════════╡
│ 640    ┆ 42      ┆ 120301 ┆ 0.00001   ┆ … ┆ 100 ┆ 2       ┆ 7.034378  ┆ 11.608908 │
│ 610    ┆ 37      ┆ 42     ┆ 0.00001   ┆ … ┆ 100 ┆ 17      ┆ 0.480504  ┆ -6.86719  │
│ 370    ┆ 2       ┆ 1029   ┆ 0.000001  ┆ … ┆ 100 ┆ 21      ┆ 7.433303  ┆ 19.779247 │
│ 510    ┆ 1       ┆ 42     ┆ 0.000001  ┆ … ┆ 100 ┆ 23      ┆ 9.841413  ┆ 22.249264 │
│ 300    ┆ 47      ┆ 2025   ┆ 0.0000001 ┆ … ┆ 100 ┆ 31      ┆ 6.580911  ┆ 15.987182 │
│ …      ┆ …       ┆ …      ┆ …         ┆ … ┆ …   ┆ …       ┆ …         ┆ …         │
│ 600    ┆ 45      ┆ 2025   ┆ 0.0000

In [7]:


plot = interactive_from_embeddings(
    embedding=umap_df[["UMAP1", "UMAP2"]].to_numpy(),
    labels=umap_df["k"],
    hover_data=umap_df.drop("UMAP1", "UMAP2"),
    point_size=3,
)

show(plot)